# Text Mining Project - Final Model
**Spring Semester 2025/2026**

Group 11:
- Ana Macedo
- Carlota Pires
- Francisca Calçoa
- Francisca Martins



<hr>
<a class="anchor" id="one-bullet"> 
<d style="color:black;">

# 1. Imports and Load Data
</a> 
</d>   

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [2]:
train = pd.read_csv('../Datasets/train.csv')
test = pd.read_csv('../Datasets/test.csv')

X_train_raw = train['text'].astype(str)
y_train_raw = train['label']

X_test_raw = test['text'].astype(str)

<hr>
<a class="anchor" id="two-bullet"> 
<d style="color:black;">

# 2. Final Model Pipeline
</a> 
</d>   


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("finiteautomata/bertweet-base-sentiment-analysis")

model = AutoModelForSequenceClassification.from_pretrained(
    "finiteautomata/bertweet-base-sentiment-analysis",
    num_labels=train['label'].nunique()
)


class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(X_train_raw, y_train_raw)
test_dataset = TweetDataset(X_test_raw)

training_args = TrainingArguments(
    output_dir="./bertweet-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()




[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
50,0.841798
100,0.627963
150,0.488666
200,0.442084
250,0.434614
300,0.424302
350,0.397719
400,0.420201
450,0.408128
500,0.349821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2985, training_loss=0.20936872479903637, metrics={'train_runtime': 221.3767, 'train_samples_per_second': 215.538, 'train_steps_per_second': 13.484, 'total_flos': 2476562752765710.0, 'train_loss': 0.20936872479903637, 'epoch': 5.0})

### Export Test Predictions

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(axis=1)

submission = pd.DataFrame({
    "id": test["id"],
    "label": pred_labels
})

submission.to_csv("pred_11.csv", index=False)